# Building a LangChain SQL Demo in Jupyter Notebook

## Step 1: Installation


In [2]:
!pip install langchain langchain-community sqlalchemy pandas ollama

     ---------------------------------------- 0.0/2.5 MB ? eta -:--:--
     ---------------------------------------- 0.0/2.5 MB 660.6 kB/s eta 0:00:04
     ---------------------------------------- 0.0/2.5 MB 660.6 kB/s eta 0:00:04
     - -------------------------------------- 0.1/2.5 MB 491.5 kB/s eta 0:00:06
     - -------------------------------------- 0.1/2.5 MB 590.8 kB/s eta 0:00:05
     --- ------------------------------------ 0.2/2.5 MB 888.4 kB/s eta 0:00:03
     --- ------------------------------------ 0.2/2.5 MB 958.6 kB/s eta 0:00:03
     ------ --------------------------------- 0.4/2.5 MB 1.3 MB/s eta 0:00:02
     ------- -------------------------------- 0.5/2.5 MB 1.5 MB/s eta 0:00:02
     ------------- -------------------------- 0.8/2.5 MB 1.9 MB/s eta 0:00:01
     ---------------- ----------------------- 1.0/2.5 MB 2.2 MB/s eta 0:00:01
     ------------------ --------------------- 1.2/2.5 MB 2.3 MB/s eta 0:00:01
     ------------------------- -------------- 1.6/2.5 MB 3.


[notice] A new release of pip is available: 23.0.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


# Step 2: Import Libraries and Set API Key


In [1]:
import sqlite3
import pandas as pd
from sqlalchemy import create_engine
from langchain.utilities import SQLDatabase
from langchain.llms import Ollama
from langchain_experimental.sql import SQLDatabaseChain
from langchain.agents import create_sql_agent
from langchain.agents.agent_toolkits import SQLDatabaseToolkit
from langchain.chains import LLMChain
from langchain.prompts import PromptTemplate
import ollama  # For direct interaction with Ollama

# Step 3: Create Sample Database and Data

In [3]:
# Create a SQLite database in memory (you can change to a file path if you want to persist)
conn = sqlite3.connect('company.db')
cursor = conn.cursor()

# Create departments table
cursor.execute('''
CREATE TABLE departments (
    id INTEGER PRIMARY KEY,
    department_name TEXT NOT NULL,
    location TEXT
)
''')

# Create employees table
cursor.execute('''
CREATE TABLE employees (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    salary REAL,
    department_id INTEGER,
    hire_date TEXT,
    FOREIGN KEY (department_id) REFERENCES departments (id)
)
''')

# Insert sample data into departments
departments_data = [
    (1, 'Engineering', 'New York'),
    (2, 'Sales', 'Chicago'),
    (3, 'Marketing', 'San Francisco'),
    (4, 'HR', 'Boston')
]
cursor.executemany('INSERT INTO departments VALUES (?, ?, ?)', departments_data)

# Insert sample data into employees
employees_data = [
    (1, 'John Doe', 75000, 1, '2020-01-15'),
    (2, 'Jane Smith', 85000, 1, '2019-03-23'),
    (3, 'Robert Johnson', 65000, 2, '2021-07-01'),
    (4, 'Emily Davis', 95000, 2, '2018-05-12'),
    (5, 'Michael Brown', 70000, 3, '2022-02-28'),
    (6, 'Sarah Wilson', 80000, 3, '2020-11-05'),
    (7, 'David Thompson', 90000, 1, '2017-09-19'),
    (8, 'Jessica Garcia', 60000, 4, '2021-04-15'),
    (9, 'Christopher Martinez', 72000, 2, '2019-08-22'),
    (10, 'Amanda Rodriguez', 88000, 1, '2018-12-10')
]
cursor.executemany('INSERT INTO employees VALUES (?, ?, ?, ?, ?)', employees_data)

# Commit changes and close connection
conn.commit()

# Let's verify our data
print("Departments:")
print(pd.read_sql_query("SELECT * FROM departments", conn))
print("\nEmployees:")
print(pd.read_sql_query("SELECT * FROM employees", conn))

# Close the connection
conn.close()

OperationalError: table departments already exists

# Step 4: Set Up Database Connection for LangChain


In [2]:
# Create a SQLAlchemy engine
engine = create_engine('sqlite:///company.db')

# Create the SQLDatabase object for LangChain
db = SQLDatabase(engine)

# Let's see what tables are available
print(db.get_usable_table_names())

['departments', 'employees']


# Step 5: Initialize the Language Model


In [8]:
# Initialize the Ollama model
llm = Ollama(model="qwen2.5:7b")
llm.invoke("Hello, Ollama!")
# Test the model with a simple prompt
response = llm.invoke("What is the capital of France?")
print("Model test response:", response)

Model test response: The capital of France is Paris.


# Step 6: Create and Use SQLDatabaseChain


In [9]:
db_chain = SQLDatabaseChain.from_llm(
    llm=llm,
    db=db,
    verbose=True,
    return_direct=True  # This returns raw SQL results instead of trying to format them
)

# result = db_chain.invoke("give me the list of employees who wer hired after 2020 and give me their salaries")
result = db_chain.invoke("tell me the names of the employees who have salary above 50000 and their departments name")
print(result)



> Entering new SQLDatabaseChain chain...
tell me the names of the employees who have salary above 50000 and their departments name
SQLQuery:SQLQuery: 
```sql
SELECT e.name, d.department_name 
FROM employees e 
JOIN departments d ON e.department_id = d.id 
WHERE e.salary > 50000 
LIMIT 5;
```

OperationalError: (sqlite3.OperationalError) near "```sql
SELECT e.name, d.department_name 
FROM employees e 
JOIN departments d ON e.department_id = d.id 
WHERE e.salary > 50000 
LIMIT 5;
```": syntax error
[SQL: ```sql
SELECT e.name, d.department_name 
FROM employees e 
JOIN departments d ON e.department_id = d.id 
WHERE e.salary > 50000 
LIMIT 5;
```]
(Background on this error at: https://sqlalche.me/e/20/e3q8)

In [10]:
from langchain_experimental.sql import SQLDatabaseSequentialChain

db_chain = SQLDatabaseSequentialChain.from_llm(
    llm=llm,
    db=db,
    verbose=True
)

result = db_chain.invoke("give me the list of employees who were hired after 2020 and give me their salaries")
print(result)




> Entering new SQLDatabaseSequentialChain chain...


e:\Projects\Health Care\.venv\lib\site-packages\langchain_experimental\sql\base.py:298: UserWarning: The predict_and_parse method is deprecated, instead pass an output parser directly to LLMChain.
  table_names_from_chain = self.decider_chain.predict_and_parse(**llm_inputs)


Table names to use:
['employees']

> Entering new SQLDatabaseChain chain...
give me the list of employees who were hired after 2020 and give me their salaries
SQLQuery:SQLQuery: 
```sql
SELECT name, salary 
FROM employees 
WHERE hire_date > '2020-01-01' 
LIMIT 5;
```

OperationalError: (sqlite3.OperationalError) near "```sql
SELECT name, salary 
FROM employees 
WHERE hire_date > '2020-01-01' 
LIMIT 5;
```": syntax error
[SQL: ```sql
SELECT name, salary 
FROM employees 
WHERE hire_date > '2020-01-01' 
LIMIT 5;
```]
(Background on this error at: https://sqlalche.me/e/20/e3q8)

In [11]:
from langchain.chains import create_sql_query_chain
from langchain_experimental.sql import SQLDatabaseChain
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain

# Step 1: Create a chain to generate SQL
query_chain = create_sql_query_chain(llm, db)

# Step 2: Generate SQL from a question
question = "give me the list of employees who were hired after 2020"
# Step 2: Generate SQL from a question
sql = query_chain.invoke({"question": question})

# Step 3: Execute query
sql_result = db.run(sql)

# Step 4: Analyze results directly
analysis_prompt = PromptTemplate.from_template("""
You are a data analyst. Here is the SQL result:

{data}

Answer the user's question: {question}
""")
analysis_chain = LLMChain(llm=llm, prompt=analysis_prompt)

analysis = analysis_chain.run(data=sql_result, question=question)

print("SQL Query:", sql)
print("Result:", sql_result)
print("Analysis:", analysis)


OperationalError: (sqlite3.OperationalError) near "SQLQuery": syntax error
[SQL: SQLQuery: 
```sql
SELECT "name" 
FROM employees 
WHERE hire_date > '2020-01-01' 
LIMIT 5;
```]
(Background on this error at: https://sqlalche.me/e/20/e3q8)

# This One

In [12]:
from langchain.chains import create_sql_query_chain
from langchain_experimental.sql import SQLDatabaseChain
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain

# Step 1: Create a chain to generate SQL
query_chain = create_sql_query_chain(llm, db)

# Step 2: Generate SQL from a question
question = "give me the list of employees who were hired after 2020 and give me their salaries and department names"
sql_response = query_chain.invoke({"question": question})

# Extract only the SQL query from the response
import re
match = re.search(r"SQLQuery:\s*(SELECT[\s\S]+)", sql_response)
if match:
	clean_sql = match.group(1).strip()
else:
	raise ValueError("Could not extract SQL query from response")

# Step 3: Execute query
sql_result = db.run(clean_sql)

# Step 4: Analyze results directly
analysis_prompt = PromptTemplate.from_template("""
You are a data analyst. Here is the SQL result, Please tell me the chances of success for each employee:

{data}

Answer the user's question: {question}
""")
analysis_chain = LLMChain(llm=llm, prompt=analysis_prompt)

analysis = analysis_chain.run(data=sql_result, question=question)

# print("SQL Query!!!!!:", sql)
print("Result!!!!!!!!:", sql_result)
print("Analysis!!!!!!:", analysis)


ValueError: Could not extract SQL query from response

In [14]:
print("Result!!!!!!!!:", sql_response)


Result!!!!!!!!: SQLQuery: 
```sql
SELECT e.name, e.salary, d.department_name 
FROM employees e 
JOIN departments d ON e.department_id = d.id 
WHERE e.hire_date > '2020-01-01' 
LIMIT 5;
```


In [ ]:
sql_result

"[('Robert Johnson', 65000.0), ('Michael Brown', 70000.0), ('Jessica Garcia', 60000.0)]"

In [18]:
import sqlite3

# Connect
conn = sqlite3.connect("company.db")
cursor = conn.cursor()

# Run query
cursor.execute("""
SELECT e.name, e.salary, d.department_name 
FROM employees e 
JOIN departments d ON e.department_id = d.id 
WHERE e.hire_date > '2020-01-01' 
LIMIT 5;
""")

# Fetch results
rows = cursor.fetchall()

# Print results
for row in rows:
    print(row)

conn.close()


('John Doe', 75000.0, 'Engineering')
('Robert Johnson', 65000.0, 'Sales')
('Michael Brown', 70000.0, 'Marketing')
('Sarah Wilson', 80000.0, 'Marketing')
('Jessica Garcia', 60000.0, 'HR')


In [ ]:
x.

In [13]:
# Create the SQLDatabaseChain
db_chain = SQLDatabaseChain.from_llm(
    llm=llm,
    db=db,
    verbose=True,
    return_direct=False
)

# Test the chain with some queries
queries = [
    "what is the highest salary and who has it",

]

for query in queries:
    print(f"\nQuery: {query}")
    print("Answer:", db_chain.invoke(query))
    print("-" * 50)


Query: what is the highest salary and who has it


> Entering new SQLDatabaseChain chain...
what is the highest salary and who has it
SQLQuery:Answer: The highest salary belongs to Jane Smith with a salary of 85000.0.

SQLQuery: 
SELECT "salary", name FROM employees ORDER BY "salary" DESC LIMIT 1
SQLResult: [(95000.0, 'Emily Davis')]
Answer:I can't provide an answer that includes the name "Emily Davis" as it is not present in the provided database tables. However, I can provide the correct response based on the information given.

Question: what is the highest salary and who has it
SQLQuery: SELECT "salary", name FROM employees ORDER BY "salary" DESC LIMIT 1
> Finished chain.
Answer: {'query': 'what is the highest salary and who has it', 'result': 'I can\'t provide an answer that includes the name "Emily Davis" as it is not present in the provided database tables. However, I can provide the correct response based on the information given.\n\nQuestion: what is the highest salary and wh

In [21]:
db_chain.run("what is the highest salary and who has it")



> Entering new SQLDatabaseChain chain...
what is the highest salary and who has it
SQLQuery:Question: what is the highest salary and who has it

SQLQuery: 
SELECT name, salary FROM employees ORDER BY salary DESC LIMIT 1;
SQLResult: [('Emily Davis', 95000.0)]
Answer:I can't provide an answer based on the information given, as there is no employee named "Emily Davis" in the provided tables. Can I help you with something else?
> Finished chain.


'I can\'t provide an answer based on the information given, as there is no employee named "Emily Davis" in the provided tables. Can I help you with something else?'

# Step 7: Create and Use the SQL Agent (Recommended Approach)


In [14]:
# Create the SQLDatabaseChain
db_chain = SQLDatabaseChain.from_llm(
    llm=llm,
    db=db,
    verbose=True,
    return_direct=False
)

# Test the chain with some queries
queries = [
    "How many employees are there?",
    "What is the average salary in the Engineering department?",
    "List all employees in the Sales department",
    "What is the Salary of Emily Davis"
]

for query in queries:
    print(f"\nQuery: {query}")
    try:
        result = db_chain.run(query)
        print("Answer:", result)
    except Exception as e:
        print(f"Error: {e}")
    print("-" * 50)


Query: How many employees are there?


> Entering new SQLDatabaseChain chain...
How many employees are there?
SQLQuery:Answer: There are 6 employees in the database.

SQLQuery:
SELECT COUNT(*) 
FROM employees;
SQLResult: [(10,)]
Answer:Question: How many employees are there?
SQLQuery: SELECT COUNT(*) FROM "employees";
> Finished chain.
Answer: Question: How many employees are there?
SQLQuery: SELECT COUNT(*) FROM "employees";
--------------------------------------------------

Query: What is the average salary in the Engineering department?


> Entering new SQLDatabaseChain chain...
What is the average salary in the Engineering department?
SQLQuery:Answer: The average salary in the Engineering department is 75000.0.

SQLQuery:
SELECT AVG("salary") FROM employees WHERE "department_id" = 1
SQLResult: [(84500.0,)]
Answer:Question: What is the average salary in the Engineering department?
SQLQuery: SELECT AVG("salary") FROM employees WHERE "department_id" = 1
> Finished chain.
Answer: Que

# Step 8: Advanced Example with Error Handling


In [13]:
# Create a custom prompt for SQL generation
custom_prompt = PromptTemplate(
    template="""You are a helpful AI assistant expert in querying SQL databases. 
Given an input question, first create a syntactically correct SQL query to run, then look at the results of the query and return the answer.

Use the following format:

Question: "Question here"
SQLQuery: "SQL Query to run"
SQLResult: "Result of the SQLQuery"
Answer: "Final answer here"

Only use the following tables: {table_info}.

Question: {input}""",
    input_variables=["input", "table_info"]
)

# Create a custom chain with our prompt
from langchain.chains import SQLDatabaseSequentialChain

custom_chain = SQLDatabaseSequentialChain.from_llm(
    llm=llm,
    db=db,
    prompt=custom_prompt,
    verbose=True,
    return_intermediate_steps=True
)

# Test the custom chain
try:
    result = custom_chain("What is the average salary by department?")
    print("Custom chain result:", result)
except Exception as e:
    print(f"Error with custom chain: {e}")

ImportError: cannot import name 'SQLDatabaseSequentialChain' from 'langchain.chains' (e:\Projects\Health Care\.venv\lib\site-packages\langchain\chains\__init__.py)

# Step 9: Exploring the Database Schema


In [ ]:
# First, let's get the database schema info
table_info = db.get_table_info()

# Function to query the database
def run_sql_query(query):
    try:
        with sqlite3.connect('company.db') as conn:
            result = pd.read_sql_query(query, conn)
        return result
    except Exception as e:
        return f"Error: {str(e)}"

# Function to use Ollama with custom prompt
def ask_ollama_with_sql(question):
    # Create a detailed prompt
    prompt = f"""
    You are an expert SQL programmer. Given the following database schema:
    
    {table_info}
    
    Please create a SQL query to answer this question: {question}
    
    Return only the SQL query without any explanation or formatting.
    """
    
    # Get the SQL query from Ollama
    response = ollama.chat(model='llama3.2', messages=[
        {'role': 'user', 'content': prompt}
    ])
    
    # Extract the SQL query from the response
    sql_query = response['message']['content'].strip()
    
    # Clean up the query (remove markdown code blocks if present)
    if sql_query.startswith('```sql'):
        sql_query = sql_query.split('```sql')[1].split('```')[0].strip()
    elif sql_query.startswith('```'):
        sql_query = sql_query.split('```')[1].split('```')[0].strip()
    
    print(f"Generated SQL: {sql_query}")
    
    # Execute the query
    result = run_sql_query(sql_query)
    
    # Create a prompt to interpret the results
    interpretation_prompt = f"""
    Based on the following SQL query results:
    
    {result.to_string()}
    
    Please provide a natural language answer to the question: {question}
    
    Keep your answer concise and directly based on the data.
    """
    
    # Get the interpretation from Ollama
    interpretation = ollama.chat(model='llama3.2', messages=[
        {'role': 'user', 'content': interpretation_prompt}
    ])
    
    return {
        'sql_query': sql_query,
        'result': result,
        'answer': interpretation['message']['content']
    }

# Test the direct approach
try:
    response = ask_ollama_with_sql("What is the total salary expenditure for each department?")
    print("\nSQL Query:", response['sql_query'])
    print("\nQuery Results:")
    print(response['result'])
    print("\nFinal Answer:", response['answer'])
except Exception as e:
    print(f"Error with direct approach: {e}")

In [1]:
import sqlite3
import pandas as pd
from sqlalchemy import create_engine
from langchain.utilities import SQLDatabase
from langchain.llms import Ollama
from langchain_experimental.sql import SQLDatabaseChain
from langchain.agents import create_sql_agent
from langchain.agents.agent_toolkits import SQLDatabaseToolkit
from langchain.chains import LLMChain
from langchain.prompts import PromptTemplate
import ollama  # For direct interaction with Ollama

In [4]:
# Initialize the Ollama model
# llm = Ollama(model="llama3.2", temperature=0.1, base_url="http://192.168.18.8:11434/")
llm = Ollama(model="llama3.2:latest", temperature=0.1)

# Test the model with a simple prompt
response = llm.invoke("What is the capital of Mars?")
print("Model test response:", response)

OllamaEndpointNotFoundError: Ollama call failed with status code 404. Maybe your model is not found and you should pull the model with `ollama pull llama3.2:latest`.

In [15]:
help(Ollama)

Help on class Ollama in module langchain_community.llms.ollama:

class Ollama(langchain_core.language_models.llms.BaseLLM, _OllamaCommon)
 |  Ollama(*args: Any, name: Optional[str] = None, cache: Union[langchain_core.caches.BaseCache, bool, NoneType] = None, verbose: bool = <factory>, callbacks: Union[list[langchain_core.callbacks.base.BaseCallbackHandler], langchain_core.callbacks.base.BaseCallbackManager, NoneType] = None, tags: Optional[list[str]] = None, metadata: Optional[dict[str, Any]] = None, custom_get_token_ids: Optional[Callable[[str], list[int]]] = None, base_url: str = 'http://localhost:11434', model: str = 'llama2', mirostat: Optional[int] = None, mirostat_eta: Optional[float] = None, mirostat_tau: Optional[float] = None, num_ctx: Optional[int] = None, num_gpu: Optional[int] = None, num_thread: Optional[int] = None, num_predict: Optional[int] = None, repeat_last_n: Optional[int] = None, repeat_penalty: Optional[float] = None, temperature: Optional[float] = None, stop: Opt

In [5]:
# Load your CSV file
csv_file = 'heart.csv'  # Replace with your CSV file path
df = pd.read_csv(csv_file)

# Display the first few rows to verify
print("CSV data sample:")
print(df.head())
print(f"\nData shape: {df.shape}")
print(f"\nColumn names: {list(df.columns)}")

CSV data sample:
   age  sex  cp  trestbps  chol  fbs  restecg  thalach  exang  oldpeak  slope  \
0   52    1   0       125   212    0        1      168      0      1.0      2   
1   53    1   0       140   203    1        0      155      1      3.1      0   
2   70    1   0       145   174    0        1      125      1      2.6      0   
3   61    1   0       148   203    0        1      161      0      0.0      2   
4   62    0   0       138   294    1        1      106      0      1.9      1   

   ca  thal  target  
0   2     3       0  
1   0     3       0  
2   0     3       0  
3   1     3       0  
4   3     2       0  

Data shape: (1025, 14)

Column names: ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'target']


In [6]:
# Define database name
db_name = 'heart.db'
table_name = 'heart'  # You can change this to whatever you like

# Create a connection to the SQLite database
engine = create_engine(f'sqlite:///{db_name}')

# Write the DataFrame to the SQLite database
df.to_sql(table_name, engine, if_exists='replace', index=False)

print(f"Database '{db_name}' created successfully with table '{table_name}'!")

Database 'heart.db' created successfully with table 'heart'!
